# Credit Card Default – Data Cleaning
### การตรวจสอบและทำความสะอาดข้อมูล

**English**
- **What this notebook does:** checks the raw Excel file for problems (missing values, duplicates, wrong categories, impossible numbers, outliers), then builds a clean table that the other notebooks use.
- **Input:** `Default.xlsx` (raw data, 10,000 customers). SQL checks: `sql/01_cleaning.sql`.
- **Output:** `data/default_clean.csv` – the clean data used by `02_eda`, `03_modeling` and `04_tariff_recommendation`.
- **Read it in 2 minutes:** skip the code. Read the **Result** note under each check, then the **scorecard** in Section 9.

**ภาษาไทย**
- **Notebook นี้ทำอะไร:** ตรวจไฟล์ Excel ดิบว่ามีปัญหาไหม (ค่าว่าง แถวซ้ำ หมวดหมู่ผิด ตัวเลขที่เป็นไปไม่ได้ ค่าผิดปกติ) แล้วสร้างตารางข้อมูลที่สะอาดให้ notebook อื่นใช้ต่อ
- **ข้อมูลเข้า:** `Default.xlsx` (ข้อมูลดิบ ลูกค้า 10,000 คน) คำสั่ง SQL สำหรับตรวจอยู่ใน `sql/01_cleaning.sql`
- **ผลลัพธ์:** `data/default_clean.csv` – ข้อมูลสะอาดที่ `02_eda`, `03_modeling` และ `04_tariff_recommendation` ใช้
- **อ่านจบใน 2 นาที:** ข้ามโค้ดได้เลย อ่าน **ผลลัพธ์** ใต้การตรวจแต่ละข้อ แล้วดู **ตารางสรุปผลการตรวจ (scorecard)** ในหัวข้อ 9

**Columns / คอลัมน์:** `default` = defaulted on card debt? / ผิดนัดชำระหนี้บัตรหรือไม่ · `student` = is a student? / เป็นนักศึกษาหรือไม่ · `balance` = card debt left after the monthly payment / ยอดหนี้คงค้างหลังจ่ายรายเดือน · `income` = income / รายได้

No charts here – data checks are easiest to read as small tables. / notebook นี้ไม่มีกราฟ เพราะผลการตรวจข้อมูลอ่านง่ายที่สุดในรูปตารางเล็กๆ

## 1. Load the raw Excel file / โหลดไฟล์ Excel ดิบ
Code only, you can skip. / ส่วนโค้ด ข้ามได้

In [ ]:
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from sql_utils import load_queries

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", None)

raw = pd.read_excel(ROOT / "Default.xlsx", sheet_name="Default")
con = duckdb.connect()
con.register("raw_df", raw)
con.execute("CREATE OR REPLACE TABLE raw_default AS SELECT * FROM raw_df")
queries = load_queries(ROOT / "sql" / "01_cleaning.sql")


def run(name):
    """Show the named query from sql/01_cleaning.sql, then return its result (None for CREATE)."""
    display(Markdown(f"```sql\n{queries[name]}\n```"))
    rel = con.sql(queries[name])
    return rel.df() if rel is not None else None


print(raw.shape)
raw.head()

(10000, 5)


,rownames,default,student,balance,income
0,1,No,No,729.526495,44361.625074
1,2,No,Yes,817.180407,12106.134700
2,3,No,No,1073.549164,31767.138947
3,4,No,No,529.250605,35704.493935
4,5,No,No,785.655883,38463.495879


## 2. Rows and IDs / จำนวนแถวและรหัสลูกค้า
**Why:** every customer should appear once, with their own ID. / **ทำไม:** ลูกค้าแต่ละคนควรมีแค่ 1 แถว และมีรหัสของตัวเอง

In [2]:
row_count = run("row_count")
row_count

```sql
SELECT
    COUNT(*)                 AS n_rows,
    COUNT(DISTINCT rownames) AS n_unique_ids
FROM raw_default;
```

,n_rows,n_unique_ids
0,10000,10000


**Result:** 10,000 rows and 10,000 unique IDs → every customer appears once.
**ผลลัพธ์:** 10,000 แถว และรหัสไม่ซ้ำ 10,000 รหัส → ลูกค้าแต่ละคนมีแค่ 1 แถว

## 3. Missing values / ค่าว่าง
**Why:** empty cells break models and averages. / **ทำไม:** ช่องที่ว่างทำให้โมเดลและค่าเฉลี่ยผิดพลาด

In [3]:
missing = run("missing_values")
missing

```sql
SELECT
    COUNT(*) - COUNT(rownames)  AS missing_rownames,
    COUNT(*) - COUNT("default") AS missing_default,
    COUNT(*) - COUNT(student)   AS missing_student,
    COUNT(*) - COUNT(balance)   AS missing_balance,
    COUNT(*) - COUNT(income)    AS missing_income
FROM raw_default;
```

,missing_rownames,missing_default,missing_student,missing_balance,missing_income
0,0,0,0,0,0


**Result:** 0 missing values in every column → nothing to fill in.
**ผลลัพธ์:** ไม่มีค่าว่างเลยในทุกคอลัมน์ → ไม่ต้องเติมข้อมูล

## 4. Duplicate rows / แถวซ้ำ
**Why:** the same customer counted twice would bias every result. / **ทำไม:** ถ้านับลูกค้าคนเดียวกันซ้ำ ผลลัพธ์ทุกอย่างจะเพี้ยน

In [4]:
duplicates = run("duplicate_rows")
duplicates

```sql
SELECT CAST(COALESCE(SUM(n - 1), 0) AS INTEGER) AS duplicate_rows
FROM (
    SELECT COUNT(*) AS n
    FROM raw_default
    GROUP BY "default", student, balance, income
    HAVING COUNT(*) > 1
);
```

,duplicate_rows
0,0


**Result:** 0 duplicate rows → nothing to remove.
**ผลลัพธ์:** ไม่มีแถวซ้ำ → ไม่ต้องลบอะไร

## 5. Category values / ค่าในคอลัมน์หมวดหมู่
**Why:** `default` and `student` should only contain "Yes" or "No" (no typos like "yes " or "Y"). / **ทำไม:** `default` และ `student` ควรมีแค่ "Yes" หรือ "No" (ไม่มีคำพิมพ์ผิด เช่น "yes " หรือ "Y")

In [5]:
categories = run("category_values")
categories

```sql
SELECT 'default' AS column_name, "default" AS value, COUNT(*) AS n
FROM raw_default GROUP BY "default"
UNION ALL
SELECT 'student', student, COUNT(*)
FROM raw_default GROUP BY student
ORDER BY column_name, value;
```

,column_name,value,n
0,default,No,9667
1,default,Yes,333
2,student,No,7056
3,student,Yes,2944


**Result:** only "Yes" and "No" in both columns. **333 customers defaulted (3.3%)** and **2,944 are students (29.4%)**.
**ผลลัพธ์:** มีแค่ "Yes" และ "No" ทั้งสองคอลัมน์ **ลูกค้าผิดนัด 333 คน (3.3%)** และ **เป็นนักศึกษา 2,944 คน (29.4%)**

## 6. Number ranges / ช่วงของตัวเลข
**Why:** balance and income can never be negative. / **ทำไม:** ยอดหนี้และรายได้ไม่ควรติดลบ

In [6]:
ranges = run("numeric_ranges")
ranges

```sql
SELECT
    'balance'                                AS column_name,
    ROUND(MIN(balance), 2)                   AS min,
    ROUND(MAX(balance), 2)                   AS max,
    ROUND(AVG(balance), 2)                   AS mean,
    COUNT(*) FILTER (WHERE balance < 0)      AS n_negative,
    COUNT(*) FILTER (WHERE balance = 0)      AS n_zero
FROM raw_default
UNION ALL
SELECT
    'income',
    ROUND(MIN(income), 2),
    ROUND(MAX(income), 2),
    ROUND(AVG(income), 2),
    COUNT(*) FILTER (WHERE income < 0),
    COUNT(*) FILTER (WHERE income = 0)
FROM raw_default;
```

,column_name,min,max,mean,n_negative,n_zero
0,balance,0.00,2654.32,835.37,0,499
1,income,771.97,73554.23,33516.98,0,0


**Result:** no negative values. **499 customers have a balance of exactly 0**. This is normal (for example, they pay in full every month), so we keep them.
**ผลลัพธ์:** ไม่มีค่าติดลบ **ลูกค้า 499 คนมียอดหนี้เป็น 0 พอดี** ซึ่งเป็นเรื่องปกติ (เช่น จ่ายเต็มจำนวนทุกเดือน) จึงเก็บไว้

## 7. Outliers / ค่าผิดปกติ
**Why:** very large or small values may be typing errors, or real but unusual customers. We use the common **IQR rule**: a value is an outlier if it is far outside the middle 50% of the data.

**ทำไม:** ค่าที่สูงหรือต่ำมากอาจเป็นการพิมพ์ผิด หรือเป็นลูกค้าจริงที่แปลกกว่าคนอื่น เราใช้ **กฎ IQR** ซึ่งเป็นวิธีมาตรฐาน: ค่าที่อยู่ห่างจากกลุ่มกลาง 50% ของข้อมูลมากเกินไปถือเป็นค่าผิดปกติ

In [7]:
outliers = run("outliers_iqr")
outliers

```sql
WITH q AS (
    SELECT
        QUANTILE_CONT(balance, 0.25) AS b_q1, QUANTILE_CONT(balance, 0.75) AS b_q3,
        QUANTILE_CONT(income, 0.25)  AS i_q1, QUANTILE_CONT(income, 0.75)  AS i_q3
    FROM raw_default
)
SELECT
    COUNT(*) FILTER (WHERE balance > b_q3 + 1.5 * (b_q3 - b_q1)
                        OR balance < b_q1 - 1.5 * (b_q3 - b_q1)) AS balance_outliers,
    COUNT(*) FILTER (WHERE balance > b_q3 + 1.5 * (b_q3 - b_q1)
                       AND "default" = 'Yes')                    AS balance_outliers_defaulted,
    COUNT(*) FILTER (WHERE income > i_q3 + 1.5 * (i_q3 - i_q1)
                        OR income < i_q1 - 1.5 * (i_q3 - i_q1))  AS income_outliers
FROM raw_default, q;
```

,balance_outliers,balance_outliers_defaulted,income_outliers
0,31,26,0


**Result:** 31 customers have unusually high balances, and **26 of them defaulted**. They are real high-risk customers, not typing errors, and they are exactly who the model must learn from, so we **keep** them. Income has no outliers.
**ผลลัพธ์:** ลูกค้า 31 คนมียอดหนี้สูงผิดปกติ และ **26 คนในนั้นผิดนัด** เป็นลูกค้าเสี่ยงสูงจริงๆ ไม่ใช่การพิมพ์ผิด และเป็นกลุ่มที่โมเดลต้องเรียนรู้ จึง **เก็บไว้** ส่วนรายได้ไม่มีค่าผิดปกติ

## 8. Build and check the clean table / สร้างและตรวจตารางข้อมูลสะอาด

| Raw column | Clean column | Change | การเปลี่ยนแปลง |
|---|---|---|---|
| `rownames` | `customer_id` | Renamed | เปลี่ยนชื่อ |
| `default` | `default_flag` | Yes/No → 1/0 | เปลี่ยน Yes/No เป็น 1/0 |
| `student` | `is_student` | Yes/No → 1/0 | เปลี่ยน Yes/No เป็น 1/0 |
| `balance` | `balance` | No change | ไม่เปลี่ยน |
| `income` | `income` | No change | ไม่เปลี่ยน |

**Why 1/0?** Models and SQL averages work with numbers: the average of `default_flag` is the default rate. / **ทำไมต้อง 1/0?** โมเดลและ SQL คำนวณกับตัวเลขได้ง่าย เช่น ค่าเฉลี่ยของ `default_flag` ก็คืออัตราการผิดนัด

In [8]:
run("create_clean_table")
con.sql("SELECT * FROM default_clean ORDER BY customer_id LIMIT 5").df()

```sql
-- Unexpected category values become NULL so the validation step below catches them.
CREATE OR REPLACE TABLE default_clean AS
SELECT
    CAST(rownames AS INTEGER) AS customer_id,
    CASE UPPER(TRIM("default")) WHEN 'YES' THEN 1 WHEN 'NO' THEN 0 END AS default_flag,
    CASE UPPER(TRIM(student))   WHEN 'YES' THEN 1 WHEN 'NO' THEN 0 END AS is_student,
    CAST(balance AS DOUBLE) AS balance,
    CAST(income  AS DOUBLE) AS income
FROM raw_default;
```

,customer_id,default_flag,is_student,balance,income
0,1,0,0,729.526495,44361.625074
1,2,0,1,817.180407,12106.134700
2,3,0,0,1073.549164,31767.138947
3,4,0,0,529.250605,35704.493935
4,5,0,0,785.655883,38463.495879


In [9]:
validation = run("validate_clean")
assert validation.loc[0, "n_null"] == 0, "Clean table has NULLs - check category values"
validation

```sql
SELECT
    COUNT(*) AS n_rows,
    COUNT(*) FILTER (WHERE default_flag IS NULL OR is_student IS NULL
                        OR balance IS NULL OR income IS NULL) AS n_null,
    CAST(SUM(default_flag) AS INTEGER) AS n_default,
    CAST(SUM(is_student) AS INTEGER)   AS n_student
FROM default_clean;
```

,n_rows,n_null,n_default,n_student
0,10000,0,333,2944


**Result:** 10,000 rows, 0 NULLs, 333 defaulters, 2,944 students – the same as the raw data, so nothing was lost.
**ผลลัพธ์:** 10,000 แถว ไม่มีค่าว่าง ผิดนัด 333 คน นักศึกษา 2,944 คน ตรงกับข้อมูลดิบ ไม่มีข้อมูลหาย

## 9. Scorecard and save / สรุปผลการตรวจ และบันทึกไฟล์

In [10]:
checks = [
    ("One row per customer / ลูกค้า 1 คน 1 แถว",
     f"{row_count.loc[0, 'n_rows']:,} rows, {row_count.loc[0, 'n_unique_ids']:,} unique IDs",
     row_count.loc[0, "n_rows"] == row_count.loc[0, "n_unique_ids"]),
    ("No missing values / ไม่มีค่าว่าง",
     f"{int(missing.to_numpy().sum())} missing",
     missing.to_numpy().sum() == 0),
    ("No duplicate rows / ไม่มีแถวซ้ำ",
     f"{duplicates.loc[0, 'duplicate_rows']} duplicates",
     duplicates.loc[0, "duplicate_rows"] == 0),
    ("Categories are Yes/No only / หมวดหมู่มีแค่ Yes/No",
     ", ".join(sorted(categories["value"].unique())),
     set(categories["value"]) <= {"Yes", "No"}),
    ("No negative numbers / ไม่มีค่าติดลบ",
     f"{int(ranges['n_negative'].sum())} negative",
     ranges["n_negative"].sum() == 0),
    ("No NULLs after cleaning / ไม่มีค่าว่างหลังทำความสะอาด",
     f"{validation.loc[0, 'n_null']} NULL",
     validation.loc[0, "n_null"] == 0),
]
scorecard = pd.DataFrame(checks, columns=["check", "result", "passed"])
scorecard["status"] = np.where(scorecard.pop("passed"), "PASS", "FAIL")
scorecard

,check,result,status
0,One row per customer / ลูกค้า 1 คน 1 แถว,"10,000 rows, 10,000 unique IDs",PASS
1,No missing values / ไม่มีค่าว่าง,0 missing,PASS
2,No duplicate rows / ไม่มีแถวซ้ำ,0 duplicates,PASS
3,Categories are Yes/No only / หมวดหมู่มีแค่ Yes/No,"No, Yes",PASS
4,No negative numbers / ไม่มีค่าติดลบ,0 negative,PASS
5,No NULLs after cleaning / ไม่มีค่าว่างหลังทำคว...,0 NULL,PASS


In [11]:
clean = con.sql("SELECT * FROM default_clean ORDER BY customer_id").df()
clean.to_csv(DATA_DIR / "default_clean.csv", index=False)
print(f"Saved {len(clean):,} rows to data/default_clean.csv")

Saved 10,000 rows to data/default_clean.csv


## Summary / สรุป

**EN:** the raw data was already clean – **all 6 checks PASS**. We removed nothing. The only changes were renaming `rownames` to `customer_id` and turning Yes/No into 1/0. Next step: `02_eda.ipynb`.

**TH:** ข้อมูลดิบสะอาดอยู่แล้ว – **ผ่านการตรวจทั้ง 6 ข้อ** ไม่ได้ลบข้อมูลใดออก สิ่งที่เปลี่ยนมีแค่การเปลี่ยนชื่อ `rownames` เป็น `customer_id` และเปลี่ยน Yes/No เป็น 1/0 ขั้นต่อไป: `02_eda.ipynb`